# Forecasting Botswana's Export Diversification Opportunities
## Data Collection, Feature Engineering & Post-Model Analysis Pipeline
*Boemo Seboni | DSS 680 – Summer 2026*

Diamonds account for roughly 80% of Botswana's export earnings. This project asks **which non-diamond export categories show the strongest potential for future growth**, and **how reliably predictive models can identify them**.

This notebook covers every step done in Python. Model training took place in SAS Viya, between Parts 6 and 7:

| Part | Step | Where |
|---|---|---|
| 1 | Collect Botswana's annual trade data (2000–2023) from the UN Comtrade API | Python |
| 2 | Reshape into one row per product category per year | Python |
| 3 | Separate diamonds from other precious goods in Chapter 71 | Python |
| 4 | Build and verify the master product-year table | Python |
| 5 | Engineer predictors and the prediction target | Python |
| 6 | Build and export SAS-ready development and scoring tables | Python |
| — | Train and compare eight models; score 2023 data with the champion SVM | **SAS Viya** |
| 7 | Build the recommendation table, multi-year forecasts, and a cross-check of SAS results | Python |

**Data source:** [UN Comtrade](https://comtradeplus.un.org/) public API (annual totals, HS 2-digit chapters, all trading partners combined)

**Key output files:** `botswana_complete_master_2000_2023.csv`, `botswana_sas_model_development.csv`, `botswana_sas_2024_scoring.csv`, `botswana_diversification_recommendations_final.csv`, `horizon_model_validation_summary.csv`

Forecasts are tested against official Statistics Botswana 2024–2025 figures in a separate notebook, `Statistics_Botswana_Validation_Seboni.ipynb`.

*Code developed with AI assistance.*

# Part 1 — Collect Trade Data from UN Comtrade

### 1.1 Setup
Imports libraries and sets the API endpoint, Botswana's UN country code (72), and the study period.

In [1]:
import requests
import pandas as pd
import numpy as np
import time

COMTRADE_API_URL = "https://comtradeapi.un.org/public/v1/preview/C/A/HS"

BOTSWANA_CODE = 72
START_YEAR = 2000
END_YEAR = 2023

### 1.2 Define the download function
Downloads one year of Botswana's total exports (`X`) or imports (`M`) to the world, broken down by HS 2-digit product chapter (`AG2`).

In [2]:
def get_botswana_trade_data(year, flow_code):
    """Download Botswana trade data for one year.

    flow_code:
        "X" = exports
        "M" = imports
    """
    
    params = {
        "period": year,
        "reporterCode": BOTSWANA_CODE,
        "flowCode": flow_code,
        "partnerCode": 0,
        "partner2Code": 0,
        "cmdCode": "AG2",
        "customsCode": "C00",
        "motCode": 0,
        "maxRecords": 500,
        "includeDesc": "true"
    }

    response = requests.get(COMTRADE_API_URL, params=params, timeout=120)
    response.raise_for_status()

    api_result = response.json()

    if api_result.get("error"):
        raise ValueError(f"API error for {year}: {api_result['error']}")

    return api_result["data"]

### 1.3 Collect exports and imports for 2000–2023
Loops through every year and trade flow, keeping the fields needed for analysis. A two-second pause between requests respects the API's rate limit.

In [3]:
# Collect Botswana's annual export and import data from 2000 through 2023

all_trade_records = []

for year in range(START_YEAR, END_YEAR + 1):
    for flow_code in ["X", "M"]:
        
        records = get_botswana_trade_data(year, flow_code)

        for row in records:
            all_trade_records.append({
                "reporting_year": row.get("refYear"),
                "reporting_country": row.get("reporterDesc"),
                "trade_flow_type": row.get("flowDesc"),
                "product_category_code": row.get("cmdCode"),
                "product_category_description": row.get("cmdDesc"),
                "trading_partner": row.get("partnerDesc"),
                "total_trade_value_usd": row.get("primaryValue")
            })

        print(f"Completed {year} - {'Exports' if flow_code == 'X' else 'Imports'}")
        time.sleep(2)

df_trade = pd.DataFrame(all_trade_records)

print("\nTotal records collected:", len(df_trade))
df_trade.head()

Completed 2000 - Exports
Completed 2000 - Imports
Completed 2001 - Exports
Completed 2001 - Imports
Completed 2002 - Exports
Completed 2002 - Imports
Completed 2003 - Exports
Completed 2003 - Imports
Completed 2004 - Exports
Completed 2004 - Imports
Completed 2005 - Exports
Completed 2005 - Imports
Completed 2006 - Exports
Completed 2006 - Imports
Completed 2007 - Exports
Completed 2007 - Imports
Completed 2008 - Exports
Completed 2008 - Imports
Completed 2009 - Exports
Completed 2009 - Imports
Completed 2010 - Exports
Completed 2010 - Imports
Completed 2011 - Exports
Completed 2011 - Imports
Completed 2012 - Exports
Completed 2012 - Imports
Completed 2013 - Exports
Completed 2013 - Imports
Completed 2014 - Exports
Completed 2014 - Imports
Completed 2015 - Exports
Completed 2015 - Imports
Completed 2016 - Exports
Completed 2016 - Imports
Completed 2017 - Exports
Completed 2017 - Imports
Completed 2018 - Exports
Completed 2018 - Imports
Completed 2019 - Exports
Completed 2019 - Imports


,reporting_year,reporting_country,trade_flow_type,product_category_code,product_category_description,trading_partner,total_trade_value_usd
0,2000,Botswana,Export,05,Animal originated products; not elsewhere spec...,World,2707.0
1,2000,Botswana,Export,06,"Trees and other plants, live; bulbs, roots and...",World,1058.0
2,2000,Botswana,Export,07,Vegetables and certain roots and tubers; edible,World,137067.0
3,2000,Botswana,Export,08,"Fruit and nuts, edible; peel of citrus fruit o...",World,39645.0
4,2000,Botswana,Export,09,"Coffee, tea, mate and spices",World,179957.0


### 1.4 Check coverage
Confirms how many product categories, and what total trade value, were collected for each year and trade flow.

In [4]:
END_YEAR = 2023

# Check the number of records collected for each year and trade-flow type

coverage_check = (
    df_trade
    .groupby(["reporting_year", "trade_flow_type"])
    .agg(
        number_of_product_categories=("product_category_code", "count"),
        total_trade_value_usd=("total_trade_value_usd", "sum")
    )
    .reset_index()
)

coverage_check.tail(10)

,reporting_year,trade_flow_type,number_of_product_categories,total_trade_value_usd
38,2019,Export,97,5.312907e+09
39,2019,Import,97,6.657431e+09
40,2020,Export,93,4.376456e+09
41,2020,Import,97,6.605142e+09
42,2021,Export,93,7.474892e+09
43,2021,Import,97,8.462769e+09
44,2022,Export,95,8.414814e+09
45,2022,Import,97,8.114915e+09
46,2023,Export,97,5.502717e+09
47,2023,Import,97,6.417468e+09


### 1.5 Standardize data types
Converts years and trade values to numbers and pads product codes to two digits. Without padding, codes such as `01` (live animals) lose their leading zero and become `1`. Chapter 71, which contains diamonds, is then displayed for review.

In [5]:
# Standardize data types and verify the diamond-related product category

df_trade["reporting_year"] = pd.to_numeric(
    df_trade["reporting_year"], errors="coerce"
).astype("Int64")

df_trade["product_category_code"] = (
    df_trade["product_category_code"]
    .astype(str)
    .str.zfill(2)
)

df_trade["total_trade_value_usd"] = pd.to_numeric(
    df_trade["total_trade_value_usd"], errors="coerce"
)

diamond_trade = df_trade[
    df_trade["product_category_code"] == "71"
].sort_values(["reporting_year", "trade_flow_type"])

print("Diamond/precious-stones records:", len(diamond_trade))

diamond_trade.tail(10)

Diamond/precious-stones records: 48


,reporting_year,reporting_country,trade_flow_type,product_category_code,product_category_description,trading_partner,total_trade_value_usd
3710,2019,Botswana,Export,71,"Natural, cultured pearls; precious, semi-preci...",World,4.848415e+09
3862,2019,Botswana,Import,71,"Natural, cultured pearls; precious, semi-preci...",World,1.894044e+09
3943,2020,Botswana,Export,71,"Natural, cultured pearls; precious, semi-preci...",World,3.912742e+09
4038,2020,Botswana,Import,71,"Natural, cultured pearls; precious, semi-preci...",World,2.042367e+09
4061,2021,Botswana,Export,71,"Natural, cultured pearls; precious, semi-preci...",World,6.776026e+09
4167,2021,Botswana,Import,71,"Natural, cultured pearls; precious, semi-preci...",World,2.972569e+09
4300,2022,Botswana,Export,71,"Natural, cultured pearls; precious, semi-preci...",World,7.372567e+09
4353,2022,Botswana,Import,71,"Natural, cultured pearls; precious, semi-preci...",World,2.249417e+09
4537,2023,Botswana,Export,71,"Natural, cultured pearls; precious, semi-preci...",World,4.357365e+09
4546,2023,Botswana,Import,71,"Natural, cultured pearls; precious, semi-preci...",World,9.546347e+08


### 1.6 Check for duplicate records
Confirms that each year, trade flow, and product category appears only once.

In [6]:
# Check whether any year-product-flow combination appears more than once

duplicate_mask = df_trade.duplicated(
    subset=[
        "reporting_year",
        "trade_flow_type",
        "product_category_code"
    ],
    keep=False
)

print("Duplicate records:", duplicate_mask.sum())
print("Years covered:", df_trade["reporting_year"].min(), "to", df_trade["reporting_year"].max())
print("Unique product categories:", df_trade["product_category_code"].nunique())

df_trade.loc[duplicate_mask].head()

Duplicate records: 0
Years covered: 2000 to 2023
Unique product categories: 97


,reporting_year,reporting_country,trade_flow_type,product_category_code,product_category_description,trading_partner,total_trade_value_usd


# Part 2 — Reshape into a Product-Year Table

### 2.1 One row per product category per year
Pivots the data so each row holds one product category's exports and imports for one year.

In [7]:
# Reshape to one row per year and product category

df_product_year = (
    df_trade
    .pivot(
        index=[
            "reporting_year",
            "product_category_code",
            "product_category_description"
        ],
        columns="trade_flow_type",
        values="total_trade_value_usd"
    )
    .reset_index()
    .rename(
        columns={
            "Export": "annual_export_value_usd",
            "Import": "annual_import_value_usd"
        }
    )
)

df_product_year.columns.name = None

print("Product-year rows:", len(df_product_year))
print("\nMissing values by column:")
print(df_product_year.isna().sum())

df_product_year.head()

Product-year rows: 2328

Missing values by column:
reporting_year                   0
product_category_code            0
product_category_description     0
annual_export_value_usd         20
annual_import_value_usd          0
dtype: int64


,reporting_year,product_category_code,product_category_description,annual_export_value_usd,annual_import_value_usd
0,2000,01,Animals; live,235090.0,3687201.0
1,2000,02,Meat and edible meat offal,51610108.0,1544670.0
2,2000,03,"Fish and crustaceans, molluscs and other aquat...",1534.0,2065253.0
3,2000,04,Dairy produce; birds' eggs; natural honey; edi...,77585.0,28681115.0
4,2000,05,Animal originated products; not elsewhere spec...,2707.0,314340.0


### 2.2 Flag missing export records
Twenty product-years have no recorded exports. They are flagged rather than filled in here, so the missing values can be handled explicitly during modeling in SAS Viya.

In [8]:
# Flag missing export records but preserve them for treatment in SAS Viya

df_product_year["no_recorded_exports"] = (
    df_product_year["annual_export_value_usd"].isna().astype(int)
)

print("Missing export values to address in SAS:",
      df_product_year["annual_export_value_usd"].isna().sum())

Missing export values to address in SAS: 20


# Part 3 — Separate Diamonds from Other Precious Goods

HS Chapter 71 combines diamonds with pearls, other precious stones, precious metals, and jewelry. Because diamonds are the concentration problem this project studies, not a diversification opportunity, Chapter 71 is split into **diamonds (HS 7102)** and **all other precious goods (`71_OTHER`)**.

### 3.1 Define the detailed download function
Downloads the 18 four-digit subcategories within Chapter 71, retrying with increasing waits if the API's rate limit is reached.

In [9]:
# HS4 subcategories contained within chapter 71

CHAPTER_71_PRODUCT_CODES = [
    "7101", "7102", "7103", "7104", "7105", "7106",
    "7107", "7108", "7109", "7110", "7111", "7112",
    "7113", "7114", "7115", "7116", "7117", "7118"
]

def get_precious_products_trade_data(year, flow_code):
    """Download detailed chapter 71 trade data for one year."""

    params = {
        "period": year,
        "reporterCode": BOTSWANA_CODE,
        "flowCode": flow_code,
        "partnerCode": 0,
        "partner2Code": 0,
        "cmdCode": ",".join(CHAPTER_71_PRODUCT_CODES),
        "customsCode": "C00",
        "motCode": 0,
        "maxRecords": 500,
        "includeDesc": "true"
    }

    for attempt in range(5):
        response = requests.get(
            COMTRADE_API_URL,
            params=params,
            timeout=120
        )

        if response.status_code == 429:
            wait_time = 5 * (attempt + 1)
            print(f"Rate limit reached. Waiting {wait_time} seconds...")
            time.sleep(wait_time)
            continue

        response.raise_for_status()
        api_result = response.json()

        if api_result.get("error"):
            raise ValueError(
                f"API error for {year}: {api_result['error']}"
            )

        return api_result["data"]

    raise RuntimeError(
        f"Unable to retrieve {year}, flow {flow_code}, after five attempts."
    )

### 3.2 Collect detailed Chapter 71 data for 2000–2023

In [11]:
# Collect detailed precious-stones, diamonds, metals and jewelry data

all_precious_product_records = []

for year in range(START_YEAR, END_YEAR + 1):
    for flow_code in ["X", "M"]:

        records = get_precious_products_trade_data(year, flow_code)

        for row in records:
            all_precious_product_records.append({
                "reporting_year": row.get("refYear"),
                "reporting_country": row.get("reporterDesc"),
                "trade_flow_type": row.get("flowDesc"),
                "detailed_product_code": row.get("cmdCode"),
                "detailed_product_description": row.get("cmdDesc"),
                "trading_partner": row.get("partnerDesc"),
                "total_trade_value_usd": row.get("primaryValue")
            })

        flow_name = "Exports" if flow_code == "X" else "Imports"
        print(f"Completed {year} - {flow_name}")

        time.sleep(2)

df_precious_products = pd.DataFrame(all_precious_product_records)

df_precious_products["detailed_product_code"] = (
    df_precious_products["detailed_product_code"]
    .astype(str)
    .str.zfill(4)
)

df_precious_products["total_trade_value_usd"] = pd.to_numeric(
    df_precious_products["total_trade_value_usd"],
    errors="coerce"
)

print("\nTotal detailed records:", len(df_precious_products))

df_precious_products.head()

Completed 2000 - Exports
Completed 2000 - Imports
Completed 2001 - Exports
Completed 2001 - Imports
Completed 2002 - Exports
Completed 2002 - Imports
Completed 2003 - Exports
Completed 2003 - Imports
Completed 2004 - Exports
Completed 2004 - Imports
Completed 2005 - Exports
Completed 2005 - Imports
Completed 2006 - Exports
Completed 2006 - Imports
Completed 2007 - Exports
Completed 2007 - Imports
Completed 2008 - Exports
Completed 2008 - Imports
Completed 2009 - Exports
Completed 2009 - Imports
Completed 2010 - Exports
Completed 2010 - Imports
Completed 2011 - Exports
Completed 2011 - Imports
Completed 2012 - Exports
Completed 2012 - Imports
Completed 2013 - Exports
Completed 2013 - Imports
Completed 2014 - Exports
Completed 2014 - Imports
Completed 2015 - Exports
Completed 2015 - Imports
Completed 2016 - Exports
Completed 2016 - Imports
Completed 2017 - Exports
Completed 2017 - Imports
Completed 2018 - Exports
Completed 2018 - Imports
Completed 2019 - Exports
Completed 2019 - Imports


,reporting_year,reporting_country,trade_flow_type,detailed_product_code,detailed_product_description,trading_partner,total_trade_value_usd
0,2000,Botswana,Export,7103,Precious (excluding diamond) and semi-precious...,World,143900.0
1,2000,Botswana,Export,7114,Articles of goldsmiths' or silversmiths' wares...,World,3113.0
2,2000,Botswana,Export,7108,Gold (including gold plated with platinum) unw...,World,580867.0
3,2000,Botswana,Export,7117,Imitation jewellery,World,94238.0
4,2000,Botswana,Export,7113,"Jewellery articles and parts thereof, of preci...",World,5088565.0


### 3.3 Verify the detailed data and isolate diamonds

In [12]:
# Verify detailed data quality and create the diamond-specific dataset

diamond_trade = (
    df_precious_products[
        df_precious_products["detailed_product_code"] == "7102"
    ]
    .sort_values(["reporting_year", "trade_flow_type"])
    .reset_index(drop=True)
)

print("Years covered:",
      df_precious_products["reporting_year"].min(),
      "to",
      df_precious_products["reporting_year"].max())

print("Detailed product categories:",
      df_precious_products["detailed_product_code"].nunique())

print("Missing trade values:",
      df_precious_products["total_trade_value_usd"].isna().sum())

print("Diamond-specific records:", len(diamond_trade))

diamond_trade.tail(10)

Years covered: 2000 to 2023
Detailed product categories: 18
Missing trade values: 0
Diamond-specific records: 48


,reporting_year,reporting_country,trade_flow_type,detailed_product_code,detailed_product_description,trading_partner,total_trade_value_usd
38,2019,Botswana,Export,7102,"Diamonds, whether or not worked, but not mount...",World,4.805943e+09
39,2019,Botswana,Import,7102,"Diamonds, whether or not worked, but not mount...",World,1.891958e+09
40,2020,Botswana,Export,7102,"Diamonds, whether or not worked, but not mount...",World,3.862196e+09
41,2020,Botswana,Import,7102,"Diamonds, whether or not worked, but not mount...",World,2.040547e+09
42,2021,Botswana,Export,7102,"Diamonds, whether or not worked, but not mount...",World,6.736254e+09
43,2021,Botswana,Import,7102,"Diamonds, whether or not worked, but not mount...",World,2.970179e+09
44,2022,Botswana,Export,7102,"Diamonds, whether or not worked, but not mount...",World,7.344422e+09
45,2022,Botswana,Import,7102,"Diamonds, whether or not worked, but not mount...",World,2.246130e+09
46,2023,Botswana,Export,7102,"Diamonds, whether or not worked, but not mount...",World,4.333566e+09
47,2023,Botswana,Import,7102,"Diamonds, whether or not worked, but not mount...",World,9.482597e+08


### 3.4 Save checkpoint files
Saves the raw API data and intermediate tables so the master dataset can be rebuilt without re-downloading.

In [13]:
# Save API data checkpoints before building the final SAS master dataset

df_trade.to_csv(
    "botswana_all_trade_raw_2000_2023.csv",
    index=False
)

df_product_year.to_csv(
    "botswana_all_product_year_data_2000_2023.csv",
    index=False
)

df_precious_products.to_csv(
    "botswana_precious_products_detail_2000_2023.csv",
    index=False
)

print("Three checkpoint datasets saved.")

Three checkpoint datasets saved.


### 3.5 Reshape diamond trade to one row per year

In [14]:
# Reshape diamond data to one row per year with export and import columns

df_diamond_product_year = (
    df_precious_products[
        df_precious_products["detailed_product_code"] == "7102"
    ]
    .pivot(
        index=[
            "reporting_year",
            "detailed_product_code",
            "detailed_product_description"
        ],
        columns="trade_flow_type",
        values="total_trade_value_usd"
    )
    .reset_index()
    .rename(
        columns={
            "detailed_product_code": "product_category_code",
            "detailed_product_description": "product_category_description",
            "Export": "annual_export_value_usd",
            "Import": "annual_import_value_usd"
        }
    )
)

df_diamond_product_year.columns.name = None
df_diamond_product_year["is_diamond_category"] = 1
df_diamond_product_year["no_recorded_exports"] = 0

print("Diamond product-year rows:", len(df_diamond_product_year))
print("\nMissing values:")
print(df_diamond_product_year.isna().sum())

df_diamond_product_year.tail()

Diamond product-year rows: 24

Missing values:
reporting_year                  0
product_category_code           0
product_category_description    0
annual_export_value_usd         0
annual_import_value_usd         0
is_diamond_category             0
no_recorded_exports             0
dtype: int64


,reporting_year,product_category_code,product_category_description,annual_export_value_usd,annual_import_value_usd,is_diamond_category,no_recorded_exports
19,2019,7102,"Diamonds, whether or not worked, but not mount...",4.805943e+09,1.891958e+09,1,0
20,2020,7102,"Diamonds, whether or not worked, but not mount...",3.862196e+09,2.040547e+09,1,0
21,2021,7102,"Diamonds, whether or not worked, but not mount...",6.736254e+09,2.970179e+09,1,0
22,2022,7102,"Diamonds, whether or not worked, but not mount...",7.344422e+09,2.246130e+09,1,0
23,2023,7102,"Diamonds, whether or not worked, but not mount...",4.333566e+09,9.482597e+08,1,0


### 3.6 Calculate non-diamond precious goods
Subtracts diamond trade from the Chapter 71 totals to create the `71_OTHER` category, and confirms that no subtraction produces a negative value.

In [15]:
# Separate non-diamond chapter 71 trade from diamond trade

df_chapter_71 = (
    df_product_year[
        df_product_year["product_category_code"] == "71"
    ][
        [
            "reporting_year",
            "annual_export_value_usd",
            "annual_import_value_usd"
        ]
    ]
    .rename(
        columns={
            "annual_export_value_usd": "chapter_71_export_value_usd",
            "annual_import_value_usd": "chapter_71_import_value_usd"
        }
    )
)

df_other_precious_products = df_chapter_71.merge(
    df_diamond_product_year[
        [
            "reporting_year",
            "annual_export_value_usd",
            "annual_import_value_usd"
        ]
    ].rename(
        columns={
            "annual_export_value_usd": "diamond_export_value_usd",
            "annual_import_value_usd": "diamond_import_value_usd"
        }
    ),
    on="reporting_year",
    how="left"
)

df_other_precious_products["annual_export_value_usd"] = (
    df_other_precious_products["chapter_71_export_value_usd"]
    - df_other_precious_products["diamond_export_value_usd"]
)

df_other_precious_products["annual_import_value_usd"] = (
    df_other_precious_products["chapter_71_import_value_usd"]
    - df_other_precious_products["diamond_import_value_usd"]
)

df_other_precious_products["product_category_code"] = "71_OTHER"
df_other_precious_products["product_category_description"] = (
    "Precious stones, metals and related products excluding diamonds"
)
df_other_precious_products["is_diamond_category"] = 0
df_other_precious_products["no_recorded_exports"] = 0

print(
    "Negative export differences:",
    (df_other_precious_products["annual_export_value_usd"] < 0).sum()
)

print(
    "Negative import differences:",
    (df_other_precious_products["annual_import_value_usd"] < 0).sum()
)

df_other_precious_products[
    [
        "reporting_year",
        "product_category_code",
        "annual_export_value_usd",
        "annual_import_value_usd"
    ]
].tail()

Negative export differences: 0
Negative import differences: 0


,reporting_year,product_category_code,annual_export_value_usd,annual_import_value_usd
19,2019,71_OTHER,4.247146e+07,2086261.462
20,2020,71_OTHER,5.054673e+07,1820470.484
21,2021,71_OTHER,3.977159e+07,2390107.165
22,2022,71_OTHER,2.814437e+07,3287420.810
23,2023,71_OTHER,2.379908e+07,6374929.557


# Part 4 — Build the Master Product-Year Table

### 4.1 Combine all product categories
Combines every product category except the original Chapter 71 with the new diamond and `71_OTHER` categories: **98 categories × 24 years = 2,352 rows**.

In [16]:
# Build one master table containing all product categories,
# with diamonds separated from other chapter 71 products

master_columns = [
    "reporting_year",
    "product_category_code",
    "product_category_description",
    "annual_export_value_usd",
    "annual_import_value_usd",
    "is_diamond_category",
    "no_recorded_exports"
]

# Keep every broad product category except the original combined chapter 71
df_non_chapter_71 = df_product_year[
    df_product_year["product_category_code"] != "71"
].copy()

df_non_chapter_71["is_diamond_category"] = 0

# Keep only the final columns for other precious products
df_other_precious_final = df_other_precious_products.copy()

# Combine non-chapter-71 products, diamonds, and other chapter-71 products
df_master_product_year = pd.concat(
    [
        df_non_chapter_71[master_columns],
        df_diamond_product_year[master_columns],
        df_other_precious_final[master_columns]
    ],
    ignore_index=True
).sort_values(
    ["reporting_year", "product_category_code"]
).reset_index(drop=True)

print("Master product-year rows:", len(df_master_product_year))
print("Unique product categories:",
      df_master_product_year["product_category_code"].nunique())

print("Duplicate product-year rows:",
      df_master_product_year.duplicated(
          ["reporting_year", "product_category_code"]
      ).sum())

print("\nMissing values:")
print(df_master_product_year.isna().sum())

df_master_product_year.tail()

Master product-year rows: 2352
Unique product categories: 98
Duplicate product-year rows: 0

Missing values:
reporting_year                   0
product_category_code            0
product_category_description     0
annual_export_value_usd         20
annual_import_value_usd          0
is_diamond_category              0
no_recorded_exports              0
dtype: int64


,reporting_year,product_category_code,product_category_description,annual_export_value_usd,annual_import_value_usd,is_diamond_category,no_recorded_exports
2347,2023,94,"Furniture; bedding, mattresses, mattress suppo...",3683804.681,6.300412e+07,0,0
2348,2023,95,"Toys, games and sports requisites; parts and a...",77068.157,7.368728e+06,0,0
2349,2023,96,Miscellaneous manufactured articles,107116.229,2.981888e+07,0,0
2350,2023,97,Works of art; collectors' pieces and antiques,29706.772,2.469967e+05,0,0
2351,2023,99,Commodities not specified according to kind,2152428.846,2.676573e+06,0,0


### 4.2 Reconcile against the original totals
Confirms that splitting Chapter 71 didn't add or lose any trade value. Annual export and import totals match the original data exactly.

In [17]:
# Confirm that the reconstructed master table preserves the original annual totals

original_annual_totals = (
    df_product_year
    .groupby("reporting_year")
    .agg(
        original_export_total_usd=("annual_export_value_usd", "sum"),
        original_import_total_usd=("annual_import_value_usd", "sum")
    )
    .reset_index()
)

master_annual_totals = (
    df_master_product_year
    .groupby("reporting_year")
    .agg(
        master_export_total_usd=("annual_export_value_usd", "sum"),
        master_import_total_usd=("annual_import_value_usd", "sum")
    )
    .reset_index()
)

reconciliation_check = original_annual_totals.merge(
    master_annual_totals,
    on="reporting_year"
)

reconciliation_check["export_difference_usd"] = (
    reconciliation_check["master_export_total_usd"]
    - reconciliation_check["original_export_total_usd"]
)

reconciliation_check["import_difference_usd"] = (
    reconciliation_check["master_import_total_usd"]
    - reconciliation_check["original_import_total_usd"]
)

print(
    "Largest absolute export difference:",
    reconciliation_check["export_difference_usd"].abs().max()
)

print(
    "Largest absolute import difference:",
    reconciliation_check["import_difference_usd"].abs().max()
)

reconciliation_check.tail()

Largest absolute export difference: 0.0
Largest absolute import difference: 0.0


,reporting_year,original_export_total_usd,original_import_total_usd,master_export_total_usd,master_import_total_usd,export_difference_usd,import_difference_usd
19,2019,5.312907e+09,6.657431e+09,5.312907e+09,6.657431e+09,0.0,0.0
20,2020,4.376456e+09,6.605142e+09,4.376456e+09,6.605142e+09,0.0,0.0
21,2021,7.474892e+09,8.462769e+09,7.474892e+09,8.462769e+09,0.0,0.0
22,2022,8.414814e+09,8.114915e+09,8.414814e+09,8.114915e+09,0.0,0.0
23,2023,5.502717e+09,6.417468e+09,5.502717e+09,6.417468e+09,0.0,0.0


### 4.3 Assign UN Harmonized System sections
Maps each product chapter to its official HS section (for example, Section V: Mineral products) for grouping and reporting.

In [18]:
# Assign official UN Harmonized System section codes and names

def assign_un_hs_section(product_code):
    code = str(product_code)

    # Both custom chapter 71 categories belong to official Section XIV
    if code in ["7102", "71_OTHER"]:
        return pd.Series([
            "XIV",
            "Natural or cultured pearls, precious stones and metals"
        ])

    hs2 = int(code)

    if 1 <= hs2 <= 5:
        return pd.Series(["I", "Live animals and animal products"])
    elif 6 <= hs2 <= 14:
        return pd.Series(["II", "Vegetable products"])
    elif hs2 == 15:
        return pd.Series(["III", "Animal or vegetable fats and oils"])
    elif 16 <= hs2 <= 24:
        return pd.Series(["IV", "Prepared foodstuffs, beverages and tobacco"])
    elif 25 <= hs2 <= 27:
        return pd.Series(["V", "Mineral products"])
    elif 28 <= hs2 <= 38:
        return pd.Series(["VI", "Products of the chemical or allied industries"])
    elif 39 <= hs2 <= 40:
        return pd.Series(["VII", "Plastics and rubber products"])
    elif 41 <= hs2 <= 43:
        return pd.Series(["VIII", "Raw hides, skins, leather and related articles"])
    elif 44 <= hs2 <= 46:
        return pd.Series(["IX", "Wood, cork and basketware products"])
    elif 47 <= hs2 <= 49:
        return pd.Series(["X", "Wood pulp, paper and paperboard products"])
    elif 50 <= hs2 <= 63:
        return pd.Series(["XI", "Textiles and textile articles"])
    elif 64 <= hs2 <= 67:
        return pd.Series(["XII", "Footwear, headgear and related articles"])
    elif 68 <= hs2 <= 70:
        return pd.Series(["XIII", "Stone, ceramic and glass products"])
    elif 72 <= hs2 <= 83:
        return pd.Series(["XV", "Base metals and articles of base metal"])
    elif 84 <= hs2 <= 85:
        return pd.Series(["XVI", "Machinery and electrical equipment"])
    elif 86 <= hs2 <= 89:
        return pd.Series(["XVII", "Vehicles, aircraft and transport equipment"])
    elif 90 <= hs2 <= 92:
        return pd.Series(["XVIII", "Optical, medical and precision instruments"])
    elif hs2 == 93:
        return pd.Series(["XIX", "Arms and ammunition"])
    elif 94 <= hs2 <= 96:
        return pd.Series(["XX", "Miscellaneous manufactured articles"])
    elif hs2 == 97:
        return pd.Series(["XXI", "Works of art and antiques"])
    elif hs2 == 99:
        return pd.Series([
            "UNASSIGNED",
            "Commodities not specified according to kind"
        ])

df_master_product_year[
    ["un_hs_section_code", "un_hs_section_name"]
] = df_master_product_year["product_category_code"].apply(
    assign_un_hs_section
)

# Remove the earlier unofficial grouping column if it exists
df_master_product_year.drop(
    columns=["economic_sector"],
    errors="ignore",
    inplace=True
)

print(
    "Official HS sections represented:",
    df_master_product_year.loc[
        df_master_product_year["un_hs_section_code"] != "UNASSIGNED",
        "un_hs_section_code"
    ].nunique()
)

df_master_product_year[
    [
        "product_category_code",
        "product_category_description",
        "un_hs_section_code",
        "un_hs_section_name"
    ]
].drop_duplicates().sort_values("product_category_code")

Official HS sections represented: 21


,product_category_code,product_category_description,un_hs_section_code,un_hs_section_name
0,01,Animals; live,I,Live animals and animal products
1,02,Meat and edible meat offal,I,Live animals and animal products
2,03,"Fish and crustaceans, molluscs and other aquat...",I,Live animals and animal products
3,04,Dairy produce; birds' eggs; natural honey; edi...,I,Live animals and animal products
4,05,Animal originated products; not elsewhere spec...,I,Live animals and animal products
...,...,...,...,...
1465,94,"Furniture; bedding, mattresses, mattress suppo...",XX,Miscellaneous manufactured articles
94,95,"Toys, games and sports requisites; parts and a...",XX,Miscellaneous manufactured articles
95,96,Miscellaneous manufactured articles,XX,Miscellaneous manufactured articles
96,97,Works of art; collectors' pieces and antiques,XXI,Works of art and antiques


### 4.4 Standardize product descriptions
UN Comtrade occasionally changes a category's wording between years. Each code is given its most recent description, so every code has exactly one label.

In [19]:
# Standardize each product code to its most recent available description

latest_product_descriptions = (
    df_master_product_year
    .sort_values("reporting_year")
    .drop_duplicates(
        subset="product_category_code",
        keep="last"
    )
    .set_index("product_category_code")[
        "product_category_description"
    ]
)

df_master_product_year["product_category_description"] = (
    df_master_product_year["product_category_code"]
    .map(latest_product_descriptions)
)

print(
    "Unique product codes:",
    df_master_product_year["product_category_code"].nunique()
)

print(
    "Unique code-description combinations:",
    df_master_product_year[
        [
            "product_category_code",
            "product_category_description"
        ]
    ].drop_duplicates().shape[0]
)
# Display the standardized product classification table

df_master_product_year[
    [
        "product_category_code",
        "product_category_description",
        "un_hs_section_code",
        "un_hs_section_name"
    ]
].drop_duplicates().sort_values("product_category_code").reset_index(drop=True)

Unique product codes: 98
Unique code-description combinations: 98


,product_category_code,product_category_description,un_hs_section_code,un_hs_section_name
0,01,Animals; live,I,Live animals and animal products
1,02,Meat and edible meat offal,I,Live animals and animal products
2,03,"Fish and crustaceans, molluscs and other aquat...",I,Live animals and animal products
3,04,Dairy produce; birds' eggs; natural honey; edi...,I,Live animals and animal products
4,05,Animal originated products; not elsewhere spec...,I,Live animals and animal products
...,...,...,...,...
93,94,"Furniture; bedding, mattresses, mattress suppo...",XX,Miscellaneous manufactured articles
94,95,"Toys, games and sports requisites; parts and a...",XX,Miscellaneous manufactured articles
95,96,Miscellaneous manufactured articles,XX,Miscellaneous manufactured articles
96,97,Works of art; collectors' pieces and antiques,XXI,Works of art and antiques


# Part 5 — Engineer Predictors and the Prediction Target

Every predictor uses only information available in year *t*. The target is the product's export value in year *t + 1*.

### 5.1 Create the next-year export target

In [20]:
# Create the next-year export-value prediction target

df_master_product_year = (
    df_master_product_year
    .sort_values(["product_category_code", "reporting_year"])
    .reset_index(drop=True)
)

# Use zero only for target construction when the API recorded no exports.
# The original export column remains missing for SAS imputation.
export_value_for_target = (
    df_master_product_year["annual_export_value_usd"]
    .fillna(0)
)

df_master_product_year["next_year_export_value_usd"] = (
    export_value_for_target
    .groupby(df_master_product_year["product_category_code"])
    .shift(-1)
)

df_master_product_year["target_year"] = (
    df_master_product_year["reporting_year"] + 1
)

print(
    "Rows with observed next-year targets:",
    df_master_product_year["next_year_export_value_usd"].notna().sum()
)

print(
    "Rows without targets:",
    df_master_product_year["next_year_export_value_usd"].isna().sum()
)

df_master_product_year[
    [
        "reporting_year",
        "target_year",
        "product_category_code",
        "product_category_description",
        "annual_export_value_usd",
        "next_year_export_value_usd"
    ]
].tail(10)

Rows with observed next-year targets: 2254
Rows without targets: 98


,reporting_year,target_year,product_category_code,product_category_description,annual_export_value_usd,next_year_export_value_usd
2342,2014,2015,99,Commodities not specified according to kind,284866.144,331021.287
2343,2015,2016,99,Commodities not specified according to kind,331021.287,1031430.000
2344,2016,2017,99,Commodities not specified according to kind,1031430.000,3348184.996
2345,2017,2018,99,Commodities not specified according to kind,3348184.996,4352753.643
2346,2018,2019,99,Commodities not specified according to kind,4352753.643,2593549.450
2347,2019,2020,99,Commodities not specified according to kind,2593549.450,2659142.610
2348,2020,2021,99,Commodities not specified according to kind,2659142.610,2684324.429
2349,2021,2022,99,Commodities not specified according to kind,2684324.429,2494218.540
2350,2022,2023,99,Commodities not specified according to kind,2494218.540,2152428.846
2351,2023,2024,99,Commodities not specified according to kind,2152428.846,NaN


### 5.2 Create record identifiers
Gives each row a unique ID (for example `27_2015`) and labels rows with a known next-year value for model development, and 2023 rows for future scoring.

In [21]:
# Create SAS-friendly record identifiers and data-purpose labels

df_master_product_year["product_year_id"] = (
    df_master_product_year["product_category_code"].astype(str)
    + "_"
    + df_master_product_year["reporting_year"].astype(str)
)

df_master_product_year["data_purpose"] = np.where(
    df_master_product_year["next_year_export_value_usd"].notna(),
    "MODEL_DEVELOPMENT",
    "FUTURE_SCORING"
)

print(
    "Unique record IDs:",
    df_master_product_year["product_year_id"].nunique()
)

print("\nRows by data purpose:")
print(df_master_product_year["data_purpose"].value_counts())

df_master_product_year[
    [
        "product_year_id",
        "reporting_year",
        "target_year",
        "product_category_code",
        "product_category_description",
        "data_purpose",
        "next_year_export_value_usd"
    ]
].tail(10)

Unique record IDs: 2352

Rows by data purpose:
data_purpose
MODEL_DEVELOPMENT    2254
FUTURE_SCORING         98
Name: count, dtype: int64


,product_year_id,reporting_year,target_year,product_category_code,product_category_description,data_purpose,next_year_export_value_usd
2342,99_2014,2014,2015,99,Commodities not specified according to kind,MODEL_DEVELOPMENT,331021.287
2343,99_2015,2015,2016,99,Commodities not specified according to kind,MODEL_DEVELOPMENT,1031430.000
2344,99_2016,2016,2017,99,Commodities not specified according to kind,MODEL_DEVELOPMENT,3348184.996
2345,99_2017,2017,2018,99,Commodities not specified according to kind,MODEL_DEVELOPMENT,4352753.643
2346,99_2018,2018,2019,99,Commodities not specified according to kind,MODEL_DEVELOPMENT,2593549.450
2347,99_2019,2019,2020,99,Commodities not specified according to kind,MODEL_DEVELOPMENT,2659142.610
2348,99_2020,2020,2021,99,Commodities not specified according to kind,MODEL_DEVELOPMENT,2684324.429
2349,99_2021,2021,2022,99,Commodities not specified according to kind,MODEL_DEVELOPMENT,2494218.540
2350,99_2022,2022,2023,99,Commodities not specified according to kind,MODEL_DEVELOPMENT,2152428.846
2351,99_2023,2023,2024,99,Commodities not specified according to kind,FUTURE_SCORING,NaN


### 5.3 Historical export lags
Creates export values from one to five years earlier for each product category.

In [22]:
# Create five years of historical export predictors

# Ensure every product category is ordered chronologically
df_master_product_year = (
    df_master_product_year
    .sort_values(
        ["product_category_code", "reporting_year"]
    )
    .reset_index(drop=True)
)

# Remove earlier lag columns if the previous version was already run
old_lag_columns = [
    "previous_year_export_value_usd",
    "two_years_prior_export_value_usd",
    "three_years_prior_export_value_usd",
    "export_value_lag1_usd",
    "export_value_lag2_usd",
    "export_value_lag3_usd",
    "export_value_lag4_usd",
    "export_value_lag5_usd"
]

df_master_product_year = df_master_product_year.drop(
    columns=[
        column for column in old_lag_columns
        if column in df_master_product_year.columns
    ]
)

# Use zero only while constructing historical predictors.
# The original annual_export_value_usd column remains unchanged.
export_history = (
    df_master_product_year["annual_export_value_usd"]
    .fillna(0)
)

# Create five lag variables within each product category
for lag_year in range(1, 6):
    df_master_product_year[f"export_value_lag{lag_year}_usd"] = (
        export_history
        .groupby(
            df_master_product_year["product_category_code"]
        )
        .shift(lag_year)
    )

# Check the results
lag_columns = [
    "export_value_lag1_usd",
    "export_value_lag2_usd",
    "export_value_lag3_usd",
    "export_value_lag4_usd",
    "export_value_lag5_usd"
]

print("Missing values in historical predictors:")
print(df_master_product_year[lag_columns].isna().sum())

display(
    df_master_product_year[
        [
            "reporting_year",
            "product_category_code",
            "annual_export_value_usd",
            *lag_columns,
            "next_year_export_value_usd"
        ]
    ].head(12)
)

Missing values in historical predictors:
export_value_lag1_usd     98
export_value_lag2_usd    196
export_value_lag3_usd    294
export_value_lag4_usd    392
export_value_lag5_usd    490
dtype: int64


,reporting_year,product_category_code,annual_export_value_usd,export_value_lag1_usd,export_value_lag2_usd,export_value_lag3_usd,export_value_lag4_usd,export_value_lag5_usd,next_year_export_value_usd
0,2000,01,2.350900e+05,NaN,NaN,NaN,NaN,NaN,1.247375e+06
1,2001,01,1.247375e+06,235090.000,NaN,NaN,NaN,NaN,3.356750e+05
2,2002,01,3.356750e+05,1247375.000,235090.0,NaN,NaN,NaN,4.176400e+04
3,2003,01,4.176400e+04,335675.000,1247375.0,235090.0,NaN,NaN,1.189100e+04
4,2004,01,1.189100e+04,41764.000,335675.0,1247375.0,235090.0,NaN,4.319641e+06
5,2005,01,4.319641e+06,11891.000,41764.0,335675.0,1247375.0,235090.0,8.027510e+05
6,2006,01,8.027510e+05,4319641.000,11891.0,41764.0,335675.0,1247375.0,9.454600e+04
7,2007,01,9.454600e+04,802751.000,4319641.0,11891.0,41764.0,335675.0,4.972800e+05
8,2008,01,4.972800e+05,94546.000,802751.0,4319641.0,11891.0,41764.0,5.495800e+04
9,2009,01,5.495800e+04,497280.000,94546.0,802751.0,4319641.0,11891.0,1.276605e+05


### 5.4 Growth, scale, and stability predictors
Adds each product's share of total exports, three- and five-year average exports, one-year log growth, five-year volatility, and the number of the last five years with recorded exports.

In [23]:
# Create export diversification, growth, and stability predictors

# Temporary zero-filled series used only for calculations
current_exports = (
    df_master_product_year["annual_export_value_usd"]
    .fillna(0)
)

# Botswana's total merchandise exports in each reporting year
df_master_product_year["total_exports_all_products_usd"] = (
    current_exports
    .groupby(df_master_product_year["reporting_year"])
    .transform("sum")
)

# Each product's percentage share of Botswana's annual merchandise exports
df_master_product_year["export_share_pct"] = (
    current_exports
    / df_master_product_year["total_exports_all_products_usd"]
    * 100
)

# Trailing averages include the current reporting year
df_master_product_year["export_avg_3yr_usd"] = (
    pd.concat(
        [
            current_exports,
            df_master_product_year["export_value_lag1_usd"],
            df_master_product_year["export_value_lag2_usd"]
        ],
        axis=1
    )
    .mean(axis=1)
)

df_master_product_year["export_avg_5yr_usd"] = (
    pd.concat(
        [
            current_exports,
            df_master_product_year["export_value_lag1_usd"],
            df_master_product_year["export_value_lag2_usd"],
            df_master_product_year["export_value_lag3_usd"],
            df_master_product_year["export_value_lag4_usd"]
        ],
        axis=1
    )
    .mean(axis=1)
)

# Log growth handles very large values and zero-export years
df_master_product_year["export_log_growth_1yr"] = (
    np.log1p(current_exports)
    - np.log1p(df_master_product_year["export_value_lag1_usd"])
)

# Five-year export volatility using the current year and four prior years
df_master_product_year["export_volatility_5yr_usd"] = (
    pd.concat(
        [
            current_exports,
            df_master_product_year["export_value_lag1_usd"],
            df_master_product_year["export_value_lag2_usd"],
            df_master_product_year["export_value_lag3_usd"],
            df_master_product_year["export_value_lag4_usd"]
        ],
        axis=1
    )
    .std(axis=1, ddof=1)
)

# Count how many of the trailing five years had recorded exports
recorded_export_indicator = (
    1 - df_master_product_year["no_recorded_exports"]
)

df_master_product_year["recorded_export_years_5yr"] = (
    recorded_export_indicator
    .groupby(df_master_product_year["product_category_code"])
    .rolling(window=5, min_periods=5)
    .sum()
    .reset_index(level=0, drop=True)
)

# Review missing values in the new predictors
new_predictor_columns = [
    "total_exports_all_products_usd",
    "export_share_pct",
    "export_avg_3yr_usd",
    "export_avg_5yr_usd",
    "export_log_growth_1yr",
    "export_volatility_5yr_usd",
    "recorded_export_years_5yr"
]

print("Missing values in new economic predictors:")
print(df_master_product_year[new_predictor_columns].isna().sum())

display(
    df_master_product_year[
        [
            "reporting_year",
            "product_category_code",
            "annual_export_value_usd",
            *new_predictor_columns
        ]
    ].head(10)
)

Missing values in new economic predictors:
total_exports_all_products_usd      0
export_share_pct                    0
export_avg_3yr_usd                  0
export_avg_5yr_usd                  0
export_log_growth_1yr              98
export_volatility_5yr_usd          98
recorded_export_years_5yr         392
dtype: int64


,reporting_year,product_category_code,annual_export_value_usd,total_exports_all_products_usd,export_share_pct,export_avg_3yr_usd,export_avg_5yr_usd,export_log_growth_1yr,export_volatility_5yr_usd,recorded_export_years_5yr
0,2000,01,235090.0,2.778832e+09,0.008460,2.350900e+05,2.350900e+05,NaN,NaN,NaN
1,2001,01,1247375.0,2.533037e+09,0.049244,7.412325e+05,7.412325e+05,1.668825,7.157936e+05,NaN
2,2002,01,335675.0,2.745131e+09,0.012228,6.060467e+05,6.060467e+05,-1.312651,5.576790e+05,NaN
3,2003,01,41764.0,3.801646e+09,0.001099,5.416047e+05,4.649760e+05,-2.084088,5.356687e+05,NaN
4,2004,01,11891.0,3.510762e+09,0.000339,1.297767e+05,3.743590e+05,-1.256193,5.062242e+05,5.0
5,2005,01,4319641.0,4.430628e+09,0.097495,1.457765e+06,1.191269e+06,5.895062,1.818942e+06,5.0
6,2006,01,802751.0,4.506182e+09,0.017814,1.711428e+06,1.102344e+06,-1.682882,1.826366e+06,5.0
7,2007,01,94546.0,5.072523e+09,0.001864,1.738979e+06,1.054119e+06,-2.138949,1.854636e+06,5.0
8,2008,01,497280.0,4.950877e+09,0.010044,4.648590e+05,1.145222e+06,1.660058,1.802943e+06,5.0
9,2009,01,54958.0,3.455712e+09,0.001590,2.155947e+05,1.153835e+06,-2.202568,1.796265e+06,5.0


### 5.5 Trade-position predictors
Adds import share, trade balance, a trade balance scaled from −1 to +1, and an indicator for whether the product is a net export.

In [24]:
# Create trade-position and import-competition predictors

current_exports = (
    df_master_product_year["annual_export_value_usd"]
    .fillna(0)
)

current_imports = (
    df_master_product_year["annual_import_value_usd"]
    .fillna(0)
)

# Botswana's total merchandise imports for each year
df_master_product_year["total_imports_all_products_usd"] = (
    current_imports
    .groupby(df_master_product_year["reporting_year"])
    .transform("sum")
)

# Product category's share of Botswana's annual imports
df_master_product_year["import_share_pct"] = (
    current_imports
    / df_master_product_year["total_imports_all_products_usd"]
    * 100
)

# Positive values indicate a trade surplus for the product category
df_master_product_year["product_trade_balance_usd"] = (
    current_exports - current_imports
)

# Scale-independent trade balance ranging from -1 to +1
trade_total = current_exports + current_imports

df_master_product_year["normalized_trade_balance"] = np.where(
    trade_total > 0,
    (current_exports - current_imports) / trade_total,
    0
)

# Binary indicator: 1 when exports exceed imports
df_master_product_year["is_net_exporter"] = (
    current_exports > current_imports
).astype(int)

trade_predictor_columns = [
    "total_imports_all_products_usd",
    "import_share_pct",
    "product_trade_balance_usd",
    "normalized_trade_balance",
    "is_net_exporter"
]

print("Missing values in trade-position predictors:")
print(df_master_product_year[trade_predictor_columns].isna().sum())

print(
    "\nNormalized trade-balance range:",
    df_master_product_year["normalized_trade_balance"].min(),
    "to",
    df_master_product_year["normalized_trade_balance"].max()
)

display(
    df_master_product_year[
        [
            "reporting_year",
            "product_category_code",
            "annual_export_value_usd",
            "annual_import_value_usd",
            *trade_predictor_columns
        ]
    ].head(10)
)

Missing values in trade-position predictors:
total_imports_all_products_usd    0
import_share_pct                  0
product_trade_balance_usd         0
normalized_trade_balance          0
is_net_exporter                   0
dtype: int64

Normalized trade-balance range: -1.0 to 0.9998339787902163


,reporting_year,product_category_code,annual_export_value_usd,annual_import_value_usd,total_imports_all_products_usd,import_share_pct,product_trade_balance_usd,normalized_trade_balance,is_net_exporter
0,2000,01,235090.0,3687201.0,2.078936e+09,0.177360,-3452111.0,-0.880126,0
1,2001,01,1247375.0,3459959.0,1.810783e+09,0.191075,-2212584.0,-0.470029,0
2,2002,01,335675.0,6496838.0,3.373769e+09,0.192569,-6161163.0,-0.901742,0
3,2003,01,41764.0,3926661.0,3.964050e+09,0.099057,-3884897.0,-0.978952,0
4,2004,01,11891.0,3765035.0,3.235331e+09,0.116372,-3753144.0,-0.993703,0
5,2005,01,4319641.0,4517167.0,3.162337e+09,0.142843,-197526.0,-0.022353,0
6,2006,01,802751.0,3253980.0,3.053298e+09,0.106573,-2451229.0,-0.604238,0
7,2007,01,94546.0,5073112.0,3.986916e+09,0.127244,-4978566.0,-0.963409,0
8,2008,01,497280.0,6442626.0,5.211097e+09,0.123633,-5945346.0,-0.856690,0
9,2009,01,54958.0,6812710.0,4.728029e+09,0.144092,-6757752.0,-0.983995,0


### 5.6 Compare candidate targets
Creates alternative versions of the target and compares their distributions. Raw export values are extremely right-skewed, so **next year's log export value** was selected as the modeling target.

In [25]:
# Create alternative next-year targets for SAS evaluation

# Next-year percentage share of Botswana's merchandise exports
df_master_product_year["next_year_export_share_pct"] = (
    df_master_product_year["export_share_pct"]
    .groupby(df_master_product_year["product_category_code"])
    .shift(-1)
)

# Log-transformed next-year export value
# log1p permits zero export values
df_master_product_year["next_year_log_export_value"] = (
    np.log1p(
        df_master_product_year["next_year_export_value_usd"]
    )
)

target_columns = [
    "next_year_export_value_usd",
    "next_year_log_export_value",
    "next_year_export_share_pct"
]

# Compare target distributions using model-development rows only
target_summary = (
    df_master_product_year.loc[
        df_master_product_year["data_purpose"] == "MODEL_DEVELOPMENT",
        target_columns
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "std",
            "min",
            "max",
            "skew"
        ]
    )
    .T
)

print("Target-distribution comparison:")
display(target_summary)

print("\nMissing values in candidate targets:")
print(df_master_product_year[target_columns].isna().sum())

display(
    df_master_product_year[
        [
            "reporting_year",
            "target_year",
            "product_category_code",
            "next_year_export_value_usd",
            "next_year_log_export_value",
            "next_year_export_share_pct",
            "data_purpose"
        ]
    ].tail(10)
)

Target-distribution comparison:


,count,mean,median,std,min,max,skew
next_year_export_value_usd,2254.0,5.598808e+07,706503.243000,4.783644e+08,0.0,7.344422e+09,11.422312
next_year_log_export_value,2254.0,1.310091e+01,13.468084,3.326665e+00,0.0,2.271721e+01,-0.623196
next_year_export_share_pct,2254.0,1.020408e+00,0.012960,8.096959e+00,0.0,9.062155e+01,9.754094



Missing values in candidate targets:
next_year_export_value_usd    98
next_year_log_export_value    98
next_year_export_share_pct    98
dtype: int64


,reporting_year,target_year,product_category_code,next_year_export_value_usd,next_year_log_export_value,next_year_export_share_pct,data_purpose
2342,2014,2015,99,331021.287,12.709941,0.005230,MODEL_DEVELOPMENT
2343,2015,2016,99,1031430.000,13.846458,0.014089,MODEL_DEVELOPMENT
2344,2016,2017,99,3348184.996,15.023929,0.056150,MODEL_DEVELOPMENT
2345,2017,2018,99,4352753.643,15.286319,0.064795,MODEL_DEVELOPMENT
2346,2018,2019,99,2593549.450,14.768538,0.048816,MODEL_DEVELOPMENT
2347,2019,2020,99,2659142.610,14.793515,0.060760,MODEL_DEVELOPMENT
2348,2020,2021,99,2684324.429,14.802940,0.035911,MODEL_DEVELOPMENT
2349,2021,2022,99,2494218.540,14.729486,0.029641,MODEL_DEVELOPMENT
2350,2022,2023,99,2152428.846,14.582108,0.039116,MODEL_DEVELOPMENT
2351,2023,2024,99,NaN,NaN,NaN,FUTURE_SCORING


### 5.7 Assign chronological modeling roles
Splits the data by time instead of at random, so models are always validated on years that come after the years they were trained on:

| Role | Years | Purpose |
|---|---|---|
| Insufficient history | 2000–2004 | Used only to build the five-year lags |
| Train | 2005–2018 | Model training (1,372 rows) |
| Validate | 2019–2022 | Model comparison (392 rows) |
| Future score | 2023 | Forecast 2024 exports (98 rows) |

In [26]:
# Define chronological modeling and future-scoring roles

conditions = [
    df_master_product_year["reporting_year"] <= 2004,
    
    df_master_product_year["reporting_year"].between(
        2005, 2018
    ),
    
    df_master_product_year["reporting_year"].between(
        2019, 2022
    ),
    
    df_master_product_year["reporting_year"] == 2023
]

role_labels = [
    "INSUFFICIENT_HISTORY",
    "TRAIN",
    "VALIDATE",
    "FUTURE_SCORE"
]

df_master_product_year["modeling_role"] = np.select(
    conditions,
    role_labels,
    default="REVIEW"
)

# Rows eligible for SAS model development
df_master_product_year["modeling_eligible"] = (
    df_master_product_year["modeling_role"]
    .isin(["TRAIN", "VALIDATE"])
    .astype(int)
)

print("Rows by modeling role:")
print(
    df_master_product_year["modeling_role"]
    .value_counts()
    .reindex(role_labels)
)

# Confirm that eligible rows have complete historical predictors
required_history_columns = [
    "export_value_lag1_usd",
    "export_value_lag2_usd",
    "export_value_lag3_usd",
    "export_value_lag4_usd",
    "export_value_lag5_usd",
    "export_log_growth_1yr",
    "recorded_export_years_5yr"
]

eligible_rows = (
    df_master_product_year["modeling_eligible"] == 1
)

print("\nMissing historical predictors among eligible rows:")
print(
    df_master_product_year.loc[
        eligible_rows,
        required_history_columns
    ].isna().sum()
)

print("\nYears contained in each role:")
display(
    df_master_product_year
    .groupby("modeling_role")
    .agg(
        first_reporting_year=("reporting_year", "min"),
        last_reporting_year=("reporting_year", "max"),
        number_of_rows=("product_year_id", "count"),
        number_of_products=("product_category_code", "nunique")
    )
    .reindex(role_labels)
)

Rows by modeling role:
modeling_role
INSUFFICIENT_HISTORY     490
TRAIN                   1372
VALIDATE                 392
FUTURE_SCORE              98
Name: count, dtype: int64

Missing historical predictors among eligible rows:
export_value_lag1_usd        0
export_value_lag2_usd        0
export_value_lag3_usd        0
export_value_lag4_usd        0
export_value_lag5_usd        0
export_log_growth_1yr        0
recorded_export_years_5yr    0
dtype: int64

Years contained in each role:


,first_reporting_year,last_reporting_year,number_of_rows,number_of_products
modeling_role,,,,
INSUFFICIENT_HISTORY,2000,2004,490,98
TRAIN,2005,2018,1372,98
VALIDATE,2019,2022,392,98
FUTURE_SCORE,2023,2023,98,98


### 5.8 Five-year trend and volatility on a log scale
Calculates each product's five-year export trend (the slope of log exports) and log-scale volatility. Unlike dollar-based measures, these can be compared across product categories of very different sizes.

In [27]:
# Create scale-independent five-year export trend and volatility measures

# Five-year window ordered from oldest year to current year:
# t-4, t-3, t-2, t-1, t
export_window_5yr = pd.concat(
    [
        df_master_product_year["export_value_lag4_usd"],
        df_master_product_year["export_value_lag3_usd"],
        df_master_product_year["export_value_lag2_usd"],
        df_master_product_year["export_value_lag1_usd"],
        df_master_product_year["annual_export_value_usd"].fillna(0)
    ],
    axis=1
)

export_window_5yr.columns = [
    "year_t_minus_4",
    "year_t_minus_3",
    "year_t_minus_2",
    "year_t_minus_1",
    "year_t"
]

# Apply log1p so zero export values remain valid
log_export_window_5yr = np.log1p(export_window_5yr)

# Linear trend slope across the five-year log-export history
# Positive = upward trend; negative = downward trend
trend_weights = np.array([-2, -1, 0, 1, 2])

df_master_product_year["export_log_trend_5yr"] = (
    log_export_window_5yr
    .mul(trend_weights, axis=1)
    .sum(axis=1)
    / 10
)

# Scale-independent volatility of log export values
df_master_product_year["export_log_volatility_5yr"] = (
    log_export_window_5yr.std(axis=1, ddof=1)
)

# Require all five historical observations
complete_five_year_window = export_window_5yr.notna().all(axis=1)

df_master_product_year.loc[
    ~complete_five_year_window,
    [
        "export_log_trend_5yr",
        "export_log_volatility_5yr"
    ]
] = np.nan

print("Missing values in five-year trend measures:")
print(
    df_master_product_year[
        [
            "export_log_trend_5yr",
            "export_log_volatility_5yr"
        ]
    ].isna().sum()
)

print("\nEligible-row missing values:")
print(
    df_master_product_year.loc[
        df_master_product_year["modeling_eligible"] == 1,
        [
            "export_log_trend_5yr",
            "export_log_volatility_5yr"
        ]
    ].isna().sum()
)

display(
    df_master_product_year[
        [
            "reporting_year",
            "product_category_code",
            "annual_export_value_usd",
            "export_log_growth_1yr",
            "export_log_trend_5yr",
            "export_log_volatility_5yr",
            "recorded_export_years_5yr",
            "modeling_role"
        ]
    ].head(10)
)

Missing values in five-year trend measures:
export_log_trend_5yr         392
export_log_volatility_5yr    392
dtype: int64

Eligible-row missing values:
export_log_trend_5yr         0
export_log_volatility_5yr    0
dtype: int64


,reporting_year,product_category_code,annual_export_value_usd,export_log_growth_1yr,export_log_trend_5yr,export_log_volatility_5yr,recorded_export_years_5yr,modeling_role
0,2000,01,235090.0,NaN,NaN,NaN,NaN,INSUFFICIENT_HISTORY
1,2001,01,1247375.0,1.668825,NaN,NaN,NaN,INSUFFICIENT_HISTORY
2,2002,01,335675.0,-1.312651,NaN,NaN,NaN,INSUFFICIENT_HISTORY
3,2003,01,41764.0,-2.084088,NaN,NaN,NaN,INSUFFICIENT_HISTORY
4,2004,01,11891.0,-1.256193,-0.936495,1.827457,5.0,INSUFFICIENT_HISTORY
5,2005,01,4319641.0,5.895062,-0.085602,2.411745,5.0,TRAIN
6,2006,01,802751.0,-1.682882,0.638267,2.344667,5.0,TRAIN
7,2007,01,94546.0,-2.138949,0.584626,2.359117,5.0,TRAIN
8,2008,01,497280.0,1.660058,0.364475,2.240557,5.0,TRAIN
9,2009,01,54958.0,-2.202568,-0.920757,1.747216,5.0,TRAIN


# Part 6 — Build and Export the SAS-Ready Tables

### 6.1 Build the development and scoring tables
Creates the partition indicator SAS Viya uses (`_PartInd_`: 0 = train, 1 = validate), so SAS keeps the chronological split instead of partitioning at random. Then builds:
- **Development table:** 1,764 rows × 31 columns (identifiers, predictors, the partition indicator, and the target)
- **Scoring table:** 98 rows × 29 columns of 2023 data (the same identifiers and predictors)

In [28]:
# Build final SAS-ready development and scoring tables

df_master_product_year["_PartInd_"] = np.select(
    [
        df_master_product_year["modeling_role"] == "TRAIN",
        df_master_product_year["modeling_role"] == "VALIDATE"
    ],
    [0, 1],
    default=np.nan
)

df_master_product_year["exclude_from_recommendation"] = (
    df_master_product_year["is_diamond_category"].eq(1)
    | df_master_product_year["product_category_code"].eq("99")
).astype(int)

# Variables retained in both SAS datasets
sas_shared_columns = [
    # Identifiers and labels
    "product_year_id",
    "reporting_year",
    "product_category_code",
    "product_category_description",
    "un_hs_section_code",
    "un_hs_section_name",

    # Current trade measures
    "annual_export_value_usd",
    "annual_import_value_usd",
    "total_exports_all_products_usd",
    "total_imports_all_products_usd",
    "export_share_pct",
    "import_share_pct",
    "product_trade_balance_usd",
    "normalized_trade_balance",

    # Historical predictors
    "export_value_lag1_usd",
    "export_value_lag2_usd",
    "export_value_lag3_usd",
    "export_value_lag4_usd",
    "export_value_lag5_usd",
    "export_avg_3yr_usd",
    "export_avg_5yr_usd",
    "export_log_growth_1yr",
    "export_volatility_5yr_usd",
    "export_log_trend_5yr",
    "export_log_volatility_5yr",
    "recorded_export_years_5yr",

    # Binary predictors
    "is_diamond_category",
    "no_recorded_exports",
    "is_net_exporter"
]

# Development table includes partition and primary target
df_sas_model = (
    df_master_product_year.loc[
        df_master_product_year["modeling_role"]
        .isin(["TRAIN", "VALIDATE"]),
        [
            *sas_shared_columns,
            "_PartInd_",
            "next_year_log_export_value"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

df_sas_model["_PartInd_"] = (
    df_sas_model["_PartInd_"].astype(int)
)

# Scoring table contains only identifiers and predictors
df_sas_score = (
    df_master_product_year.loc[
        df_master_product_year["modeling_role"] == "FUTURE_SCORE",
        sas_shared_columns
    ]
    .copy()
    .reset_index(drop=True)
)

print("Model-development rows and columns:", df_sas_model.shape)
print("Future-scoring rows and columns:", df_sas_score.shape)

print("\nPartition counts:")
print(df_sas_model["_PartInd_"].value_counts().sort_index())

print("\nDevelopment missing values:")
missing_model = df_sas_model.isna().sum()
print(missing_model[missing_model > 0])

print("\nScoring missing values:")
missing_score = df_sas_score.isna().sum()
print(missing_score[missing_score > 0])

print("\nColumn names exceeding 32 characters:")
print([
    column
    for column in df_sas_model.columns
    if len(column) > 32
])

Model-development rows and columns: (1764, 31)
Future-scoring rows and columns: (98, 29)

Partition counts:
_PartInd_
0    1372
1     392
Name: count, dtype: int64

Development missing values:
annual_export_value_usd    17
dtype: int64

Scoring missing values:
Series([], dtype: int64)

Column names exceeding 32 characters:
[]


### 6.2 Final checks and CSV export
Asserts that IDs are unique, the train and validation years are correct, and the scoring columns match the development columns, then saves the three files.

In [29]:
# Final validation and CSV export

assert df_master_product_year["product_year_id"].is_unique
assert df_sas_model["product_year_id"].is_unique
assert df_sas_score["product_year_id"].is_unique

assert df_sas_model["next_year_log_export_value"].notna().all()
assert set(df_sas_model["_PartInd_"].unique()) == {0, 1}

assert (
    df_sas_model.loc[
        df_sas_model["_PartInd_"] == 0,
        "reporting_year"
    ].min()
    == 2005
)

assert (
    df_sas_model.loc[
        df_sas_model["_PartInd_"] == 0,
        "reporting_year"
    ].max()
    == 2018
)

assert (
    df_sas_model.loc[
        df_sas_model["_PartInd_"] == 1,
        "reporting_year"
    ].min()
    == 2019
)

assert (
    df_sas_model.loc[
        df_sas_model["_PartInd_"] == 1,
        "reporting_year"
    ].max()
    == 2022
)

assert df_sas_score["reporting_year"].eq(2023).all()

# Scoring columns must match development inputs,
# excluding the partition variable and target
expected_scoring_columns = [
    column
    for column in df_sas_model.columns
    if column not in [
        "_PartInd_",
        "next_year_log_export_value"
    ]
]

assert (
    df_sas_score.columns.tolist()
    == expected_scoring_columns
)

df_master_product_year.to_csv(
    "botswana_complete_master_2000_2023.csv",
    index=False
)

df_sas_model.to_csv(
    "botswana_sas_model_development.csv",
    index=False
)

df_sas_score.to_csv(
    "botswana_sas_2024_scoring.csv",
    index=False
)

print("Files successfully created:")
print(
    "1. botswana_complete_master_2000_2023.csv",
    df_master_product_year.shape
)
print(
    "2. botswana_sas_model_development.csv",
    df_sas_model.shape
)
print(
    "3. botswana_sas_2024_scoring.csv",
    df_sas_score.shape
)

Files successfully created:
1. botswana_complete_master_2000_2023.csv (2352, 38)
2. botswana_sas_model_development.csv (1764, 31)
3. botswana_sas_2024_scoring.csv (98, 29)


# Modeling in SAS Viya (outside this notebook)

The development table was imported into SAS Viya Model Studio, using the pre-built chronological partition, and processed in a pipeline:
1. **Transformation node:** log-transformed 11 dollar-denominated predictors
2. **Imputation node:** set the 17 remaining missing current-year export values to 0
3. **Eight models compared by validation ASE:** linear, stepwise, and LASSO regression, decision tree, random forest, gradient boosting, neural network, and SVM
4. **Champion:** SVM (validation ASE 2.758, R² 0.814), with LASSO (ASE 2.772) used as an interpretable cross-check

The champion model scored the 2023 data, and the scored results were exported from SAS as `table_FINALPROJECTSCOREDATA.csv`, which Part 7 uses.

# Part 7 — Recommendations, Multi-Year Forecasts & Cross-Check

### 7.1 Load the master table and the SAS-scored results

In [30]:
import pandas as pd
import numpy as np
from sklearn.svm import LinearSVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

master = pd.read_csv('botswana_complete_master_2000_2023.csv', dtype={'product_category_code': str})
master = master.sort_values(['product_category_code', 'reporting_year']).reset_index(drop=True)

sas_scored = pd.read_csv('table_FINALPROJECTSCOREDATA.csv', dtype={'product_category_code': str})

### 7.2 Prepare the 2024 forecast fields
Converts SAS's log-scale predictions back into dollars, excludes diamonds and the unclassified category (`99`) from recommendations, and flags products in the top 25% for five-year volatility.

In [31]:
sas_scored['predicted_growth_2024'] = sas_scored['P_next_year_log_export_value'] - sas_scored['LOG_annual_export_value_usd']
sas_scored['predicted_2024_usd'] = np.expm1(sas_scored['P_next_year_log_export_value'])
sas_scored['exclude_from_recommendation'] = (
    (sas_scored['is_diamond_category'] == 1) | (sas_scored['product_category_code'] == '99')
).astype(int)

vol_threshold = sas_scored['export_log_volatility_5yr'].quantile(0.75)
sas_scored['volatility_flag'] = np.where(
    sas_scored['export_log_volatility_5yr'] >= vol_threshold, '⚠ High volatility', ''
)

### 7.3 Multi-year forecasts (2024–2028)
SAS scored one year ahead. To look further out, a linear SVM with the same settings as the SAS champion is trained separately for each horizon (1 to 5 years ahead), each with its own chronological validation split.

In [32]:
df = master.copy()
df['annual_export_value_usd_filled'] = df['annual_export_value_usd'].fillna(0)

log_vars_raw = [
    'annual_export_value_usd_filled', 'annual_import_value_usd',
    'export_avg_3yr_usd', 'export_avg_5yr_usd',
    'export_value_lag1_usd', 'export_value_lag2_usd', 'export_value_lag3_usd',
    'export_value_lag4_usd', 'export_value_lag5_usd',
    'total_exports_all_products_usd', 'total_imports_all_products_usd'
]
for v in log_vars_raw:
    df[f'LOG_{v}'] = np.log1p(df[v])

predictors = [
    'LOG_annual_export_value_usd_filled', 'LOG_annual_import_value_usd',
    'LOG_export_avg_3yr_usd', 'LOG_export_avg_5yr_usd',
    'LOG_export_value_lag1_usd', 'LOG_export_value_lag2_usd', 'LOG_export_value_lag3_usd',
    'LOG_export_value_lag4_usd', 'LOG_export_value_lag5_usd',
    'LOG_total_exports_all_products_usd', 'LOG_total_imports_all_products_usd',
    'export_share_pct', 'import_share_pct',
    'export_log_growth_1yr', 'export_log_trend_5yr', 'export_log_volatility_5yr',
    'normalized_trade_balance', 'product_trade_balance_usd',
    'recorded_export_years_5yr',
    'is_diamond_category', 'is_net_exporter', 'no_recorded_exports'
]

MIN_TRAIN_YEAR, MAX_DATA_YEAR = 2005, 2023
results_summary = []
predictions_by_horizon = {}

for h in range(1, 6):
    d = df.copy()
    d[f'log_export_lead{h}'] = (
        d.groupby('product_category_code')['annual_export_value_usd_filled']
        .shift(-h).pipe(np.log1p)
    )
    usable = d[
        (d['reporting_year'] >= MIN_TRAIN_YEAR) & (d['reporting_year'] <= MAX_DATA_YEAR - h) &
        d[predictors].notna().all(axis=1) & d[f'log_export_lead{h}'].notna()
    ].copy()

    years = sorted(usable['reporting_year'].unique())
    n_val = max(2, round(len(years) * 0.22))
    train = usable[usable['reporting_year'].isin(years[:-n_val])]
    validate = usable[usable['reporting_year'].isin(years[-n_val:])]

    model = make_pipeline(StandardScaler(), LinearSVR(C=0.1, max_iter=10000, dual=True))
    model.fit(train[predictors], train[f'log_export_lead{h}'])
    val_pred = model.predict(validate[predictors])

    results_summary.append({
        'horizon_years': h, 'target_calendar_year': 2023 + h,
        'n_train': len(train), 'n_validate': len(validate),
        'validate_ase': round(mean_squared_error(validate[f'log_export_lead{h}'], val_pred), 4),
        'validate_r2': round(r2_score(validate[f'log_export_lead{h}'], val_pred), 4)
    })

    score_rows = d[(d['reporting_year'] == 2023) & d[predictors].notna().all(axis=1)].copy()
    score_rows[f'pred_log_export_h{h}'] = model.predict(score_rows[predictors])
    predictions_by_horizon[h] = score_rows.set_index('product_category_code')[f'pred_log_export_h{h}']

horizon_summary = pd.DataFrame(results_summary)
horizon_summary

,horizon_years,target_calendar_year,n_train,n_validate,validate_ase,validate_r2
0,1,2024,1372,392,2.8718,0.8063
1,2,2025,1274,392,3.7393,0.7477
2,3,2026,1176,392,4.5512,0.6930
3,4,2027,1176,294,4.3480,0.6957
4,5,2028,1078,294,4.7388,0.6683


### 7.4 Cross-check the Python model against SAS
Compares the Python one-year forecasts with the SAS champion's 2024 forecasts. A correlation of 0.9995 confirms that the Python version reproduces the SAS model, which supports using it for the longer horizons.

In [33]:
sas_pred_2024 = sas_scored.set_index('product_category_code')['P_next_year_log_export_value']
py_pred_2024 = predictions_by_horizon[1]

check = pd.DataFrame({'SAS': sas_pred_2024, 'Python': py_pred_2024}).dropna()
r, _ = pearsonr(check['SAS'], check['Python'])
print(f"SAS vs. independent Python replica — correlation: r = {r:.4f}  (n={len(check)})")

SAS vs. independent Python replica — correlation: r = 0.9995  (n=98)


### 7.5 Final recommendation table
Ranks eligible product categories by predicted 2024 growth and keeps the top 10. Only categories with at least $100,000 in 2023 exports are included, so tiny categories can't dominate on a low base. Forecast growth for 2025–2028 is shown alongside, relative to 2023.

In [34]:
final = sas_scored.set_index('product_category_code').copy()

# Use actual dollar values (already log1p) to get true % growth, not log-difference
final['predicted_growth_2024'] = (
    np.expm1(final['P_next_year_log_export_value']) - final['annual_export_value_usd']
) / final['annual_export_value_usd']

for h in range(1, 6):
    if h == 1:
        continue  # already set above from the SAS champion model
    final[f'pred_growth_{2023+h}_vs_2023'] = (
        np.expm1(predictions_by_horizon[h]) - final['annual_export_value_usd']
    ) / final['annual_export_value_usd']

recommend = final[final['exclude_from_recommendation'] == 0]
credible = recommend[recommend['annual_export_value_usd'] >= 100_000].copy()

top10 = credible.sort_values('predicted_growth_2024', ascending=False).head(10).reset_index()
top10['Product'] = top10['product_category_description'].str.slice(0, 35)

display_cols = {
    'product_category_code': 'Code', 'Product': 'Product', 'volatility_flag': 'Flag',
    'annual_export_value_usd': "'23 ($)", 'predicted_2024_usd': "'24 ($)",
    'predicted_growth_2024': "'24 Growth",
    'pred_growth_2025_vs_2023': "'25", 'pred_growth_2026_vs_2023': "'26",
    'pred_growth_2027_vs_2023': "'27", 'pred_growth_2028_vs_2023': "'28",
}
final_table = top10[list(display_cols.keys())].rename(columns=display_cols)

final_table.style.format({
    "'23 ($)": '{:,.0f}', "'24 ($)": '{:,.0f}',
    "'24 Growth": '{:.1%}', "'25": '{:.1%}', "'26": '{:.1%}', "'27": '{:.1%}', "'28": '{:.1%}',
})

,Code,Product,Flag,'23 ($),'24 ($),'24 Growth,'25,'26,'27,'28
0,81,"Metals; n.e.c., cermets and article",⚠ High volatility,"309,286","743,888",140.5%,10.5%,18.1%,1.7%,24.8%
1,96,Miscellaneous manufactured articles,,"107,116","196,391",83.3%,82.9%,105.2%,116.9%,111.1%
2,10,Cereals,,"839,478","1,527,147",81.9%,92.8%,89.2%,110.5%,113.2%
3,17,Sugars and sugar confectionery,,"332,216","561,834",69.1%,94.3%,63.5%,91.5%,109.8%
4,69,Ceramic products,,"125,255","173,291",38.4%,59.3%,62.6%,62.5%,56.0%
5,27,"Mineral fuels, mineral oils and pro",,"36,738,787","49,090,812",33.6%,49.7%,76.2%,46.5%,26.9%
6,22,"Beverages, spirits and vinegar",,"1,956,453","2,542,725",30.0%,49.5%,49.7%,53.4%,50.6%
7,47,Pulp of wood or other fibrous cellu,,"879,960","1,127,243",28.1%,16.8%,8.5%,-6.3%,-19.3%
8,49,"Printed books, newspapers, pictures",,"110,557","139,150",25.9%,15.9%,43.0%,21.0%,31.5%
9,82,"Tools, implements, cutlery, spoons",,"819,108","1,021,672",24.7%,12.9%,8.0%,-6.7%,-8.7%


### 7.6 Save the results

In [35]:
final_table.to_csv('botswana_diversification_recommendations_final.csv', index=False)
horizon_summary.to_csv('horizon_model_validation_summary.csv', index=False)